# Citadel T1D + PRE50M FINAL VALIDATION (one session)
Preregistered: `docs/citadel/experiments/T1D/PLAN.md` + `PRE50M_ADDENDUM.md`. Five frozen arms (A flat / B curriculum / C teacher / D scale / E repr-diagnostic) on the tiered ladder corpus, THEN the PRE50M systems certification (SCALE2 smoke, data interface, packing, buckets, throughput curve, NEXT_50M_DECISION). Run cells 0–F in order with no edits. No secrets. Checkpoints stay out of git; receipts come home in one bundle.

In [ ]:
# 0. Fresh Citadel checkout + pinned read-only Cymek runtime (public clone, no credentials)
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
SESSION = 'docs/citadel/tpu_receipts/t1d_session'
print('SESSION_DIR=' + SESSION)
EXPECTED_CYMEK_SHA = '28bf57a0d299a2c13a99fe0046616c00a1b8530c'
print('EXPECTED_CYMEK_SHA=' + EXPECTED_CYMEK_SHA)
if rt_sha != EXPECTED_CYMEK_SHA:
    raise RuntimeError('CYMEK PIN MISMATCH: runtime ' + rt_sha +
                       ' != expected ' + EXPECTED_CYMEK_SHA +
                       ' — STOP before Cell A; reconcile the pin first')

In [ ]:
# A. Complete T1D preflight (tiers, teachers, geometry, leakage, specs, tests, TPU). If NO, STOP.
import subprocess
pa = subprocess.run(['python','-m','citadel_tpu.t1d_preflight'])
assert pa.returncode == 0, 'T1D PREFLIGHT failed — READY_FOR_T1D=NO. Diagnose, do not run.'
print('READY_FOR_T1D=YES')

In [ ]:
# B. Throughput/memory/packing calibration: ONE static shape for all arms (recorded, reused on resume).
from citadel_tpu import t1d_run as t1d
import importlib
from citadel_tpu import calculator_eval as _cev
_cev = importlib.reload(_cev)
t1d = importlib.reload(t1d)
cal = t1d.calibrate(out=SESSION + '/CALIBRATION.json')
print('selected:', cal['selected'], f"{cal['selected_tokens_per_second']:.0f} tok/s")

In [ ]:
# C. Build/reuse the ~100 MB deterministic data manifest (streamed, O(1) memory).
from citadel_tpu import tiered_data as td
man = td.build_manifest(out=SESSION + '/DATA_MANIFEST.json')
print('bytes:', man['total_bytes'], 'max_row:', man['max_row_chars'])
print('leakage nonzero:', {k: v for k, v in man['leakage'].items() if v != 0} or 'NONE')

In [ ]:
# D. Execute ALL preregistered arms + PRE50M systems certification, automatically
# (resume-safe, per-arm isolation, no interaction). This cell runs T1D A–E
# AND the PRE50M phase (smoke, data interface, packing, decision).
session = t1d.run_session(SESSION)
print(session['arms'], session['labels'])

In [ ]:
# E. BOTH final conclusions (machine-evaluated; read, do not reinterpret).
import json
cs = json.load(open(SESSION + '/CROSS_ARM_SUMMARY.json'))
print('T1D_SCIENTIFIC_VERDICT:', cs['labels'])
print(json.dumps(cs['reasons'], indent=2))
lc = json.load(open(SESSION + '/LIFT_OFF_CURVES.json'))
print('FIRST_TRAIN_LIFT_TIER by arm:', json.dumps({t: v.get('first_train_lift_tier') for t, v in lc['arms'].items()}))
print('FIRST_TEST_LIFT_TIER by arm:', json.dumps({t: v.get('first_test_lift_tier') for t, v in lc['arms'].items()}))
dec = json.load(open(SESSION + '/NEXT_50M_DECISION.json'))
print('PRE50M_SYSTEM_VERDICT: READY_FOR_50M_TRAINING =', dec['ready_for_50m_training'])
print('PRE50M_BLOCKING_REASONS:', dec['blocking_reasons'])

In [ ]:
# F. Verify the bundle mechanically, then export ONE primary file.
import json
from citadel_tpu import t1d_run as t1d
print(t1d.verify_bundle(SESSION)['status'])
from google.colab import files
files.download(SESSION + '/CITADEL_T1D_RESULTS.zip')
bm = json.load(open(SESSION + '/BUNDLE_MANIFEST.json'))
print('zip bytes:', bm['zip_bytes'], 'checkpoints bundled:', bm['checkpoints_bundled'])
if not bm['checkpoints_bundled']:
    for p in bm['checkpoints']:
        files.download(p)
print('transfer the bundle back to the operator')